# 01 — Shannon Entropy

Measure uncertainty as average surprise — the first tool of information theory and the spine of everything that follows. Build the formula from the one property surprise must have, compute it by hand before calling any helper, prove the two facts every later notebook leans on (the uniform distribution is the most uncertain; a part is never more uncertain than the whole, $H(A) \le H(AB)$), and see why that second fact is exactly what the Bell state of `01/18` breaks.

**Welcome to module 02.** You have built the quantum objects — states, mixtures, distances, two systems, entanglement, channels — with a dictionary that kept a classical column alive throughout. This module builds the *information theory and geometry* that both classical and quantum optimal transport stand on: entropy, relative entropy, mutual information and transfer entropy on the classical side (`02/01`–`02/04`), the geometry of the probability simplex (`02/05`–`02/08`), then the quantum lifts — Umegaki relative entropy, quantum mutual information, negative conditional entropy, the Bures distance (`02/09`–`02/12`). We begin with the most fundamental question: how much uncertainty does a distribution carry? Claude Shannon answered it in 1948, and his answer runs through every remaining notebook.

**Prerequisites:** `01/13_pure_vs_mixed` (the eigenvalue coin; the von Neumann entropy preview), `01/16_tensor_and_partial_trace` (joint tables, marginals, the independent and the correlated-coins tables), `01/18_entanglement` (the binary entropy $h$, written there and built from surprise here; $S(A) = 1 > S(AB) = 0$ for the Bell state), `01/19_quantum_channels` (module 01 complete; the cumulative dictionary).

**What you'll be able to do**
- Explain why surprise is $-\log_2 p$ (additivity over independent events) and compute the entropy $H(p) = -\sum_x p_x \log_2 p_x$ by hand, in bits, including the $0\log 0 = 0$ convention and the binary entropy $h(p)$.
- Prove $0 \le H(p) \le \log_2 n$ with equality at a certain outcome and at the uniform distribution (Jensen), and see that entropy is concave.
- Define joint and conditional entropy, derive the chain rule $H(AB) = H(A) + H(B|A)$ by hand, and prove $H(A) \le H(AB)$ — the promise of `01/18`–`01/19` — with the coins and the independent table of `01/16` as the two extreme cases.
- Read entropy as the floor of lossless compression (Shannon's source-coding theorem, quoted) on a source you can code by hand.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from qot_course import viz
from qot_course.colors import COLORS
from qot_course.infotheory.classical import shannon_entropy

rng = np.random.default_rng(0)   # random distributions for the Jensen check of section 2
viz.use_course_style()

## 1. Surprise, and why it is a logarithm

A rare event is surprising; a certain event is not. Shannon made this precise by asking what a *surprise function* $s(p)$ — the surprise of seeing an outcome that had probability $p$ — must satisfy:

1. $s$ decreases in $p$, and $s(1) = 0$: a certain event carries no surprise.
2. **Additivity:** for two *independent* events, the surprise of seeing both is the sum of the surprises, $s(p\,q) = s(p) + s(q)$ — learning two unrelated things is learning each.

The second property forces the logarithm: a decreasing function with $s(pq) = s(p) + s(q)$ is $s(p) = -c\log p$ for some $c > 0$ — Cauchy's functional equation, where monotonicity alone pins the solution (Cover & Thomas ch. 2). Shannon (1948, §6) and Khinchin (1957) prove the matching uniqueness for the *entropy* itself from three axioms. The constant is the unit: base $2$ gives **bits**, base $e$ gives nats ($1$ nat $= 1/\ln 2 \approx 1.443$ bits). In bits, seeing a fair coin land is $1$ bit of surprise, seeing a $1$-in-$8$ event is $3$ bits.

The **entropy** of a distribution is its *average* surprise:

$$
H(p) = \sum_x p_x\,s(p_x) = -\sum_x p_x \log_2 p_x ,
$$

with the convention $0 \log 0 = 0$ (justified by $\lim_{p\to0} p\log p = 0$: an outcome that never happens contributes nothing). For a coin with $P(\text{heads}) = p$ this is the **binary entropy** that `01/18` named,

$$
h(p) = -p\log_2 p - (1-p)\log_2(1-p),
$$

$h(\tfrac12) = 1$ bit, $h(0) = h(1) = 0$. Let's compute the surprises of a few outcomes and the entropy of a coin by hand, then compare with the course helper `shannon_entropy`, and draw $h$ — and, next to it, the entropy of a *qubit*: `01/13` showed a qubit with Bloch length $|\vec r|$ has eigenvalues $\tfrac12(1 \pm |\vec r|)$, so its von Neumann entropy is $h\big(\tfrac{1+|\vec r|}{2}\big)$ — the same curve, read from the centre of the ball to its surface.

In [ ]:
def surprise_bits(p):
    return float(-np.log2(p)) + 0.0        # + 0.0: -0.0 -> 0.0 for a certain event

def entropy_by_hand(p):
    p = np.asarray(p, dtype=float)
    nz = p[p > 0]                                  # 0 log 0 = 0: drop the impossible outcomes
    return float(-(nz * np.log2(nz)).sum()) + 0.0   # + 0.0: -0.0 -> 0.0 for a certain outcome

def h(p):                                          # the binary entropy
    return entropy_by_hand([p, 1 - p])

print("surprise of an event of probability 1/2, 1/8, 0.999, 1:", [round(surprise_bits(q), 3) for q in (0.5, 0.125, 0.999, 1.0)], "bits")
print("additivity: s(1/2 * 1/8) =", surprise_bits(0.5 * 0.125), "= s(1/2) + s(1/8) =", surprise_bits(0.5) + surprise_bits(0.125))
print("1 nat =", round(1 / np.log(2), 4), "bits")
print()
for p in (0.5, 0.9, 0.999, 1.0):
    print(f"coin P(heads) = {p:<6}  H by hand = {h(p):.4f} bits   shannon_entropy = {shannon_entropy([p, 1 - p]):.4f}")
print("\n0 log 0 -> 0:  p log p at p = 1e-3, 1e-6, 1e-12:", [f"{q * np.log2(q):.2e}" for q in (1e-3, 1e-6, 1e-12)])

bias = np.linspace(0, 1, 401)
h_curve = np.array([h(b) for b in bias])
r = np.linspace(0, 1, 201)                         # Bloch length of a qubit (01/13)
S_qubit = np.array([h((1 + x) / 2) for x in r])

fig, axes = plt.subplots(1, 2, figsize=(12, 4.4))
ax = axes[0]
ax.plot(bias, h_curve, color=COLORS["source"], lw=2.5)
ax.axvline(0.5, color=COLORS["target"], ls="--", lw=1.5, label="fair coin: 1 bit")
ax.set_xlabel("coin bias  P(heads) = p"); ax.set_ylabel("entropy h(p)  (bits)")
ax.set_title("The binary entropy: maximal for a fair coin,\nzero for a certain one", pad=12); ax.legend()
ax = axes[1]
ax.plot(r, S_qubit, color=COLORS["quantum"], lw=2.5, label=r"$S(\rho) = h\!\left(\frac{1+|\vec r|}{2}\right)$")
ax.set_xlabel(r"Bloch length $|\vec r|$  (0 = $I/2$, 1 = pure)"); ax.set_ylabel("von Neumann entropy  (bits)")
ax.set_title("The same curve for a qubit (01/13):\nthe entropy of the eigenvalue coin", pad=12); ax.legend()
plt.tight_layout(); plt.show()

**Read the output and the figure.** Surprise *adds*: the $1$-in-$16$ event $\tfrac12\cdot\tfrac18$ is $1 + 3 = 4$ bits, which is why the log is the only honest choice. A certain event is $0$ bits; an event of probability $0.999$ is a tiny $\approx 0.001$. The by-hand entropy of a coin matches `shannon_entropy` for every bias, and $p\log p$ does vanish as $p \to 0$, so dropping impossible outcomes is not a trick but a limit. Left: $h(p)$ peaks at $1$ bit for the fair coin — the most unpredictable two-outcome source — and falls to $0$ at both certain ends; it is symmetric ($h(p) = h(1-p)$: relabelling heads and tails changes nothing) and concave. Right: read with `01/13`'s eigenvalues, the *same* function gives a qubit's von Neumann entropy from its Bloch length — $1$ bit at the centre $I/2$, $0$ on the surface. The quantum entropy of module 01 *is* a Shannon entropy, of the eigenvalue coin; `02/09` will make that precise for any dimension.

## 2. Two bounds and a shape: $0 \le H \le \log_2 n$, and concavity

**Lower bound.** Each term $-p_x\log_2 p_x \ge 0$ (since $p_x \le 1$), so $H \ge 0$, with equality only when every term vanishes — one outcome has probability $1$: the certain distribution.

**Upper bound.** Write $H = \sum_x p_x \log_2\tfrac{1}{p_x} = \mathbb E\big[\log_2 \tfrac1{p_X}\big]$, the expectation of a function of the random outcome. The logarithm is **concave** — every chord lies below the graph — so the average of points on the curve sits below the curve at the average point; that is Jensen's inequality, $\mathbb E[\log_2 Y] \le \log_2 \mathbb E[Y]$. With $Y = 1/p_X$, $\mathbb E[Y] = \sum_x p_x \cdot \tfrac1{p_x} = n'$, the number of outcomes with $p_x > 0$ (so $n' \le n$), so

$$
H(p) \le \log_2 n ,
$$

with equality in $\log_2 n$ exactly when $Y$ is constant on a full support — all $n$ probabilities equal: the **uniform** distribution is the most uncertain (uniform on a smaller support gives $\log_2 n'$). (`02/02` will give a second proof, from the non-negativity of relative entropy.)

**Shape.** $H$ is **concave** in $p$: mixing two distributions, $\lambda p + (1-\lambda)q$, has entropy at least the mixture of their entropies, $H(\lambda p + (1-\lambda)q) \ge \lambda H(p) + (1-\lambda)H(q)$ — forgetting *which* source you drew from can only add uncertainty. (Each term $-t\log_2 t$ is concave in $t$ — second derivative $-1/(t\ln 2) < 0$ — and a sum of concave functions is concave.) You met the quantum echo of this in `01/13`: a mixture of states is at least as mixed as its parts. Let's check all three numerically, on random distributions where nothing is built in.

In [ ]:
n = 5
print(f"log2({n}) = {np.log2(n):.4f} bits;  uniform H = {entropy_by_hand(np.full(n, 1 / n)):.4f}")
random_ps = rng.dirichlet(np.ones(n), size=2000)                     # 2000 random probability vectors on 5 outcomes (uniform over the simplex, 02/05's object)
Hs = np.array([entropy_by_hand(p) for p in random_ps])
print(f"2000 random 5-outcome distributions: min H = {Hs.min():.4f}, max H = {Hs.max():.4f}  (all <= log2 5: {np.all(Hs <= np.log2(n) + 1e-12)})")
print("the Jensen average E[1/p_X] = sum_x p_x (1/p_x) =", (random_ps[0] * (1 / random_ps[0])).sum(), "= n, on the first random p")
# equality case: a certain distribution
print("H(certain) =", entropy_by_hand([0, 0, 1, 0, 0]))
# concavity: mixtures, 2000 random pairs
p, q = random_ps[:1000], random_ps[1000:]
lam = rng.uniform(0, 1, size=1000)
mixH = np.array([entropy_by_hand(l * a + (1 - l) * b) for l, a, b in zip(lam, p, q)])
avgH = lam * np.array([entropy_by_hand(a) for a in p]) + (1 - lam) * np.array([entropy_by_hand(b) for b in q])
print(f"concavity H(mix) >= mix of H on 1000 random pairs: {np.all(mixH >= avgH - 1e-12)};  largest gap {np.max(mixH - avgH):.3f} bits, smallest {np.min(mixH - avgH):.2e}")

**Read the output.** Two thousand random points of the $5$-outcome simplex all sit below $\log_2 5 \approx 2.32$ bits, and the uniform one sits exactly on it; the Jensen bookkeeping $\sum_x p_x\cdot\tfrac1{p_x} = n$ is literally $5$; a certain distribution has $H = 0$. On a thousand random pairs the mixture's entropy never falls below the mixture of entropies — the gap would be $0$ only for $p = q$ or $\lambda \in \{0, 1\}$; on random pairs it stays strictly positive (smallest $\sim 10^{-4}$) and reaches a few tenths of a bit. Those three facts — $H \ge 0$, $H \le \log_2 n$, $H$ concave — are used on almost every page ahead.

## 3. Two variables: joint, conditional, and why a part is never more uncertain than the whole

Take a joint table $p(a, b)$ for two variables — the tables of `01/16` §4. Its **joint entropy** is the entropy of the table read as one distribution over pairs, $H(AB) = -\sum_{a,b} p(a,b)\log_2 p(a,b)$. Its **conditional entropy** is the average, over $a$, of the entropy of the column $p(b\,|\,a) = p(a,b)/p(a)$:

$$
H(B|A) = \sum_a p(a)\,H\big(p(\cdot\,|\,a)\big) = -\sum_{a,b} p(a,b)\log_2 p(b|a) \;\ge\; 0 ,
$$

non-negative because it is an average of entropies. Now the **chain rule**, by hand: inside the joint entropy write $p(a,b) = p(a)\,p(b|a)$ and split the log,

$$
H(AB) = -\sum_{a,b} p(a,b)\big[\log_2 p(a) + \log_2 p(b|a)\big]
= \underbrace{-\sum_a p(a)\log_2 p(a)}_{H(A)} \;+\; \underbrace{-\sum_{a,b} p(a,b)\log_2 p(b|a)}_{H(B|A)} ,
$$

using $\sum_b p(a,b) = p(a)$ for the first piece. So $H(AB) = H(A) + H(B|A)$, and since $H(B|A) \ge 0$:

$$
\boxed{\,H(A) \le H(AB)\,}\qquad\text{(and by symmetry } H(B) \le H(AB)\text{)} .
$$

*A part is never more uncertain than the whole* — the classical fact `01/18` and `01/19` asked you to take on trust. Equality holds exactly when $H(B|A) = 0$: every column $p(\cdot|a)$ is certain, i.e. $B$ is a function of $A$. The two tables of `01/16` are the two extreme cases: for independent variables $p(b|a) = p(b)$, so $H(B|A) = H(B)$ and $H(AB) = H(A) + H(B)$ — the whole is as uncertain as both parts together; for the correlated coins $B$ *is* $A$, so $H(B|A) = 0$ and $H(AB) = H(A) = 1$ bit — the whole is no more uncertain than one part, and no less. Let's compute all of it by hand on both tables, plus the $90\%$-agreement table in between.

In [ ]:
def joint_entropy(P):
    return entropy_by_hand(np.asarray(P).ravel())

def conditional_entropy_B_given_A(P):
    P = np.asarray(P, dtype=float)
    pA = P.sum(axis=1)
    return float(sum(pA[a] * entropy_by_hand(P[a] / pA[a]) for a in range(P.shape[0]) if pA[a] > 0))

tables = {
    "independent (0.3,0.7)x(0.4,0.6)": np.outer([0.3, 0.7], [0.4, 0.6]),
    "agree 90%":                        np.array([[0.45, 0.05], [0.05, 0.45]]),
    "correlated coins (01/16 s.5)":     np.array([[0.5, 0.0], [0.0, 0.5]]),
}
print(f"{'table':34s} {'H(A)':>7s} {'H(B)':>7s} {'H(AB)':>7s} {'H(B|A)':>8s}   chain rule   H(A) <= H(AB)")
for name, P in tables.items():
    HA, HB = entropy_by_hand(P.sum(axis=1)), entropy_by_hand(P.sum(axis=0))
    HAB, HBgA = joint_entropy(P), conditional_entropy_B_given_A(P)
    print(f"{name:34s} {HA:7.4f} {HB:7.4f} {HAB:7.4f} {HBgA:8.4f}   {np.isclose(HAB, HA + HBgA)!s:10s}   {HA <= HAB + 1e-12}")
P = tables["independent (0.3,0.7)x(0.4,0.6)"]
print("\nindependent: H(AB) == H(A) + H(B):", np.isclose(joint_entropy(P), entropy_by_hand([0.3, 0.7]) + entropy_by_hand([0.4, 0.6])))
print("coins: H(AB) == H(A) (B is a function of A):", np.isclose(joint_entropy(tables["correlated coins (01/16 s.5)"]), 1.0))
# 2000 random 2x2 tables: the inequality never fails
rand_tables = rng.dirichlet(np.ones(4), size=2000).reshape(-1, 2, 2)
viol = sum(entropy_by_hand(T.sum(axis=1)) > joint_entropy(T) + 1e-12 for T in rand_tables)
print("random 2x2 tables with H(A) > H(AB):", viol, "of 2000")
print("\nfor contrast, the Bell state of 01/18: S(A) = 1 bit, S(AB) = 0 bits -> the 'part <= whole' rule fails (02/11).")

**Read the output.** Each row obeys the chain rule to the last digit and $H(A) \le H(AB)$. The independent table has $H(AB) = H(A) + H(B) = 0.881 + 0.971$ — the parts add up; the coins have $H(AB) = H(A) = 1$ bit and $H(B|A) = 0$ — knowing $A$ tells you $B$; the $90\%$ table sits in between, $H(B|A) = 0.469$ bits of residual uncertainty about $B$ once $A$ is known. Two thousand random tables, zero violations: classically, a part cannot be more uncertain than the whole. The Bell state of `01/18` — whole pure ($S = 0$), part maximally mixed ($S = 1$) — breaks exactly the step that made this true, the positivity of $H(B|A)$: its quantum conditional entropy is $-1$ bit, and `02/11` is about what that means. You now hold the classical inequality the quantum one is measured against.

## 4. Entropy is the floor of compression

Why bits? Because entropy is not only a measure of surprise: it is the least number of binary digits per symbol that *any* lossless code can achieve on average. Give each outcome $x$ a binary codeword of length $\ell_x$ such that no codeword is a prefix of another (a **prefix code** — then a stream decodes itself without separators); such lengths must satisfy Kraft's inequality $\sum_x 2^{-\ell_x} \le 1$, and then the average length obeys $\bar\ell = \sum_x p_x\ell_x \ge H(p)$, with equality exactly when $\ell_x = -\log_2 p_x$ — *codeword length = surprise*. Conversely, lengths $\lceil -\log_2 p_x\rceil$ always satisfy Kraft and give $\bar\ell < H(p) + 1$; coding long blocks drives the overhead to zero. That is Shannon's **source-coding theorem** (1948; Cover & Thomas ch. 5), which we quote — the proof of "$\ge$" uses the relative entropy of `02/02`.

Let's see it on a source where surprises are whole numbers of bits — $p = (\tfrac12, \tfrac14, \tfrac18, \tfrac18)$, the "dyadic" case — so the optimal code is readable by hand: codewords $0$, $10$, $110$, $111$, lengths $1, 2, 3, 3$ — each exactly its surprise. We encode a long sample and count bits per symbol.

In [ ]:
p = np.array([0.5, 0.25, 0.125, 0.125])
code_dyadic = {0: "0", 1: "10", 2: "110", 3: "111"}            # lengths 1, 2, 3, 3 = -log2 p
lengths = np.array([len(code_dyadic[x]) for x in range(4)])
print("surprises -log2 p:", -np.log2(p), "   codeword lengths:", lengths)
print("Kraft sum 2^-l:", (2.0 ** -lengths).sum(), "   average length:", (p * lengths).sum(), "   H(p) =", entropy_by_hand(p), "bits")

N = 100_000
sample = rng.choice(4, size=N, p=p)
bitstring = "".join(code_dyadic[x] for x in sample)
print(f"encoded {N} symbols into {len(bitstring)} bits = {len(bitstring) / N:.4f} bits/symbol   (H = {entropy_by_hand(p):.4f})")

# no separators needed: decode the first bits greedily -- a codeword is complete as soon as it matches
inv = {v: k for k, v in code_dyadic.items()}
decoded, buf = [], ""
for bit in bitstring[:24]:
    buf += bit
    if buf in inv:
        decoded.append(inv[buf]); buf = ""
print("first 24 bits:", bitstring[:24], " -> symbols", decoded, "  match the sample:", decoded == list(sample[:len(decoded)]))
# the same code on a source it was not designed for: longer than that source's entropy
q = np.array([0.25, 0.25, 0.25, 0.25])
print(f"the uniform source with the same code: {(q * lengths).sum():.3f} bits/symbol  >  H(q) = {entropy_by_hand(q):.3f}  (a 2-bit code would be optimal)")
# and a fixed 2-bit code on the dyadic source: worse than H(p)
print(f"the dyadic source with a fixed 2-bit code: 2.000 bits/symbol  >  H(p) = {entropy_by_hand(p):.3f}")

**Read the output.** The four surprises are exactly $1, 2, 3, 3$ bits, the codeword lengths match them, Kraft's sum is exactly $1$ (no slack), and the average length equals $H(p) = 1.75$ bits — the floor, reached. The greedy decode of the first $24$ bits recovers the first symbols of the sample with no separators — the prefix property at work. A hundred thousand encoded symbols land at $1.746$ bits per symbol — a hair *below* the floor, as a finite sample may be: the theorem bounds the *expected* length, and the sample mean fluctuates by about $\pm 0.003$ here. Use the same code on a source it was not built for and you pay: the uniform source costs $2.25$ bits per symbol against its entropy of $2$; conversely a fixed $2$-bit code on the dyadic source wastes $0.25$ bits per symbol over $H(p)$. Entropy is the price of describing a source, in bits, and a mismatched model always overpays — the extra bits are the relative entropy of `02/02`.

## Your turn

1. **A fair die, a loaded die.** Predict $H$ for a fair six-sided die ($\log_2 6$? why is that the Jensen bound with equality?), then compute it with `entropy_by_hand([1/6]*6)` and `shannon_entropy`. Then load it, `[0.9, 0.02, 0.02, 0.02, 0.02, 0.02]`: does $H$ rise or fall? Compute the surprise of each face and say which face carries the most *surprise* and which the most *average* contribution $-p\log_2 p$ — they differ.
2. **Where is the maximum, and how flat is the top?** For three outcomes $(p_1, p_2, 1 - p_1 - p_2)$, plot $H$ over the triangle (a grid in $p_1, p_2$) as a heatmap and confirm the uniform $(\tfrac13, \tfrac13, \tfrac13)$ wins with $\log_2 3$; then show that $H$ is *concave* along any straight line through the triangle by plotting it along three lines of your choice.
3. **The chain rule the other way, and a bound.** On the $90\%$-agreement table compute $H(A|B)$ and check $H(AB) = H(B) + H(A|B)$ too; then verify $H(AB) \le H(A) + H(B)$ (subadditivity) on $2000$ random tables and identify the tables where it is an equality (hint: `01/16` §4). The gap $H(A) + H(B) - H(AB)$ is the mutual information of `02/03`.
4. **Harder: a qubit's entropy and the binary entropy.** Prove from `01/13` that a qubit with Bloch vector $\vec r$ has $S(\rho) = h\big(\tfrac{1+|\vec r|}{2}\big)$, then express it as a function of the purity $\mathcal P = \tfrac12(1 + |\vec r|^2)$ and plot $S$ against $\mathcal P$; check with `von_neumann_entropy` from `qot_course.quantum.density` on a few states from `01/13`. Then the other direction: show that for a *classical* (diagonal) $\rho = \mathrm{diag}(p)$, $S(\rho) = H(p)$ exactly, and *check numerically* — on $|+\rangle$ (diagonal $(\tfrac12,\tfrac12)$, $H = 1$, $S = 0$) and on a thousand random qubit states — that for a non-diagonal $\rho$, $S(\rho) \le H(\text{diagonal of }\rho)$: coherence can only *lower* the entropy below that of the Born probabilities. (Hint for later: for a qubit $\mathrm{diag}(\rho) = \tfrac12(\rho + Z\rho Z)$, so once `02/09` gives you concavity of $S$ the proof is one line; in general it is Schur–Horn majorization, Nielsen & Chuang §11.3.3.)

## What you built

- You derived the logarithm from additivity of independent surprises, defined $H(p)$ as average surprise in bits (with $0\log 0 = 0$ as a limit), computed it by hand against the helper, and defined the binary entropy $h(p)$ — whose curve is also a qubit's von Neumann entropy read off its Bloch length.
- You proved $0 \le H \le \log_2 n$ (Jensen, equality at the uniform) and concavity, and checked them on random distributions.
- You defined joint and conditional entropy, derived the chain rule by hand, and proved $H(A) \le H(AB)$ — the classical "part $\le$ whole" rule — with the independent table and the coins as the two extremes, and the Bell state as the quantum exception waiting in `02/11`.
- You saw entropy as the floor of lossless compression: surprise = codeword length, Kraft, $1.75$ bits per symbol reached on a dyadic source, and the cost of a mismatched code.

The first stone of the module's spine is laid, and it is the same stone you stood on in `01/13` and `01/18`. Every information measure ahead — relative entropy, mutual information, transfer entropy, and their quantum lifts — is built from this one idea.

**Dictionary so far** (module 01's table, plus today's row — the classical column is earned today; its quantum partner was previewed in `01/13` and `01/18` and is earned in `02/09`):

| Classical | Quantum |
|-----------|---------|
| bit ($0$ or $1$) | qubit $a_0\,|0\rangle + a_1\,|1\rangle$ (two complex amplitudes) |
| probabilities only add (non-negative) | amplitudes add *and cancel* — interference; a relative phase $e^{i\varphi}$ is physical, a global phase is not |
| a bit's state space: two isolated points | a pure qubit's state space: the Bloch **sphere** (poles = the classical bit; equator = equal superpositions) |
| outcome probabilities $p_x$ | **Born rule** $p_x = \lvert a_x\rvert^2$ — the *shadow* one measurement casts of a state; a different measurement casts a different shadow |
| overlap of two probability vectors, $\sum_x \sqrt{p_x q_x}$ (Bhattacharyya coefficient) | **inner product** $\langle\varphi|\psi\rangle$ — the same sum with amplitudes in place of $\sqrt{p_x}$; its modulus squared is the yes-probability of "are you $|\varphi\rangle$?" (fidelity, `01/14`) |
| a reversible operation on a bit is a permutation (identity, NOT); a general operation on probability vectors is a stochastic matrix, usually lossy | a **gate** is a **unitary** matrix $U$ ($U^\dagger U = I$): preserves every inner product, is undone by $U^\dagger$, and rotates the Bloch sphere rigidly (half-angle in the matrix, full angle on the sphere); the general, lossy map is the quantum channel of `01/19` |
| a quantity is a function of the bit, $f(0), f(1)$; its average is $\sum_x p_x f(x)$ | an **observable** is a Hermitian matrix $A = A^\dagger$; its average is $\langle\psi|A|\psi\rangle$ (real), the outcomes are its eigenvalues; the Paulis are gates *and* observables, and $(\langle X\rangle,\langle Y\rangle,\langle Z\rangle)$ is the Bloch vector (measured shot by shot in `01/10`) |
| there is one way to read a bit: look at it | a **measurement basis** is any orthonormal pair $\{|b_0\rangle,|b_1\rangle\}$, $P(k) = |\langle b_k|\psi\rangle|^2$; the hardware natively reads only $Z$, so every other basis is reached by *rotate, then read* — $H$ for $X$, $S^\dagger$ then $H$ for $Y$ (the recombination and $Y$-type rules of `01/02`) |
| a bit has no continuous knob: you flip it or you do not | **rotation gates** $R_x, R_y, R_z$ (and phase gates $P(\lambda)$) are continuous dials — angle $\theta$ in the matrix as $\theta/2$, full $\theta$ on the sphere; any pure state is $R_z(\phi)R_y(\theta)|0\rangle$, any single-qubit gate is $e^{i\alpha}R_z R_y R_z$ (Nielsen & Chuang Thm 4.1) |
| the average $\sum_x p_x f(x)$ is estimated from $N$ samples with a standard error $\sqrt{\mathrm{Var}(f)/N}$ | the **expectation value** $\langle A\rangle = \langle\psi|A|\psi\rangle = \sum_k \lambda_k P(k)$ is the average of the eigenvalue outcomes; from $N$ shots of a $\pm1$ observable it is estimated with standard error $\sqrt{(1 - \langle A\rangle^2)/N} \le 1/\sqrt N$; three Pauli averages determine the state (tomography, `01/15`) |
| a randomized program on bits: initialize, apply logic gates in order, read the output; run it many times and estimate the output frequencies, with sampling error | a **circuit**: qubits initialized to $|0\rangle$, unitary gates in time order, measurement into classical bits; $N$ **shots** give **counts**, a sample of the Born shadow with the error bar above; the device runs a compiled **native** version ($R_z$, $\sqrt X$, $X$, one fixed entangler), equal to yours up to a global phase; a *systematic* miss that does not shrink with $N$ is the signature of noise, and needs $\rho$ (`01/12`) |
| probability vector $p = (p_0, p_1)$: non-negative entries summing to $1$; averages are $\sum_x p_x f(x)$; a fair coin is $(\tfrac12,\tfrac12)$ | **density matrix** $\rho = |\psi\rangle\langle\psi|$ (pure case): Hermitian, unit trace, positive; diagonal = the Born probabilities $p$, off-diagonal = the **coherence** $\tfrac12\sin\theta\,e^{-i\phi}$ (modulus: how much superposition; phase: the relative phase); $P(b) = \langle b|\rho|b\rangle$, $\langle A\rangle = \mathrm{tr}(\rho A)$, gates act as $U\rho U^\dagger$; $\rho = \tfrac12(I + \vec r\cdot\vec\sigma)$ packs the Bloch vector; a global phase drops out; a $\rho$ that is diagonal in a shared basis *is* a classical $p$ — and the mixtures of `01/13` are the $\rho$'s that are no vector's outer product |
| not knowing which bit you hold is *the only* kind of uncertainty: a coin $p$; a fair coin $(\tfrac12,\tfrac12)$ is as uncertain as it gets, and a mixture of coins is a coin | a **mixed state** $\rho = \sum_i p_i|\psi_i\rangle\langle\psi_i|$ is the *second* kind of uncertainty (ignorance of which state), distinct from superposition (a definite state with random outcomes); its Bloch vector is the average of the parts' — inside the sphere, $I/2$ at the centre; **purity** $\mathrm{tr}(\rho^2) = \tfrac12(1 + |\vec r|^2) \in [\tfrac12, 1]$, eigenvalues $\tfrac12(1 \pm |\vec r|)$, von Neumann entropy = entropy of the eigenvalue coin (preview of `02/01`, `02/09`); a mixture forgets how it was prepared; **the seed:** $|+\rangle$ and $I/2$ share the diagonal $(\tfrac12,\tfrac12)$ and differ only in the corners |
| distances between probability vectors: **total variation** $\tfrac12\sum_x|p_x - q_x|$ (the best one-shot guess between $p$ and $q$ succeeds with probability $\tfrac12(1 + \mathrm{TV})$) and the **Bhattacharyya overlap** $\sum_x\sqrt{p_x q_x}$ | **trace distance** $T = \tfrac12\|\rho - \sigma\|_1 = \tfrac12|\vec r - \vec s|$ for qubits — half the distance in the Bloch **ball** (surface = pure, depth = mixedness); Helstrom: best one-shot guess succeeds with probability $\tfrac12(1 + T)$, by measuring the eigenbasis of $\rho - \sigma$; **fidelity** $F = (\mathrm{tr}\sqrt{\sqrt\rho\sigma\sqrt\rho})^2$ — *this course's convention*: $F = |\langle\varphi|\psi\rangle|^2 = \tfrac12(1 + \vec r\cdot\vec s)$ for pure states, $\langle\psi|\sigma|\psi\rangle$ for one pure, (Bhattacharyya)$^2$ for diagonal states; $F_U = \sqrt F$ is the Uhlmann fidelity of the Bures distance (`02/12`); $1 - \sqrt F \le T \le \sqrt{1 - F}$ |
| estimating a probability vector from data: count frequencies, quote $\pm\sqrt{p(1-p)/N}$; a miscalibrated counter biases every frequency by the same factor, and no number of samples fixes it — you calibrate the counter | **state tomography**: estimate $\langle X\rangle, \langle Y\rangle, \langle Z\rangle$ from three circuits' counts, each $\pm\sqrt{(1 - \langle\sigma\rangle^2)/N}$, and pack $\hat\rho = \tfrac12(I + \hat{\vec r}\cdot\vec\sigma)$ (project back onto the ball if noise pushes it out); purity, fidelity and trace distance read straight off $\hat\rho$; tomography reports the state *as seen through the meter* — a symmetric misassignment $\epsilon$ shrinks $\hat{\vec r}$ by $1 - 2\epsilon$, a lopsided one also shifts it (calibrate it away — exercise 2; `05/02` meets the same floor on a QPU) — while an idling qubit's *state* really does decohere, coherence as $e^{-t/T_2}$, population as $1 - e^{-t/T_1}$ (channels, `01/19`) |
| **marginal** of a joint table $p(a,b)$: sum over the other variable, $p_A(a) = \sum_b p(a,b)$ (row sums); independent variables factor, $p = p_A \otimes p_B$, and each marginal is the factor; *correlated* variables (the two matched coins) have perfectly random marginals although the pair is not random at all | two systems live on $\mathcal H_A \otimes \mathcal H_B$ (Kronecker product, index $k = 2a + b$, left factor first); a **product state** $\rho_A \otimes \rho_B$ is the independent case; the **partial trace** $\rho_A = \mathrm{tr}_B\,\rho_{AB}$, $(\rho_A)_{aa'} = \sum_b \rho_{ab,a'b}$ (replace each $2\times2$ block by its trace), is the only map that reproduces every local statistic $\mathrm{tr}(\rho_{AB}(M \otimes I)) = \mathrm{tr}(\rho_A M)$; a diagonal $\rho_{AB}$ reduces *exactly* to the row sums; a correlated joint state has **mixed** marginals — the *third* way a state can be mixed: it is part of something larger (`01/18` makes the whole pure and the parts still mixed) |
| **conditional logic** on two bits: controlled-NOT / XOR, $(a, b) \mapsto (a, a \oplus b)$ — a reversible *permutation* of the four outcomes; on a joint probability table it permutes the four entries; two independent operations, one per bit, can never make the bits agree when they did not before | **CNOT**, the $4\times4$ permutation unitary $|0\rangle\langle0|\otimes I + |1\rangle\langle1|\otimes X$ (controlled-$U$ pattern; control = left factor in this course, Qiskit's `cx(1, 0)`); it permutes the four basis vectors and, by linearity, acts on *superpositions* of the control: $\mathrm{CNOT}\,(\alpha|0\rangle + \beta|1\rangle)|0\rangle = \alpha|00\rangle + \beta|11\rangle$ — a product in, a non-product out (amplitude-matrix determinant $\ne 0$); no $M \otimes N$ can do that; CNOT$(H \otimes I)|00\rangle$ is the Bell state, whose corners are *coherence* between $|00\rangle$ and $|11\rangle$ (a product $|{+}{+}\rangle$ has them too; the coins have none) — entanglement proper is `01/18` |
| a joint table whose whole is *certain* (one outcome, probability $1$) has certain marginals; mixed marginals force a mixed whole (the coins); the correlation of two bits is fully described by their joint table, and any table is a mixture of point masses — every classical correlation is a *shared randomness* | **entanglement**: a *pure* joint state that is not a product (amplitude-matrix $\det\Psi \ne 0$) has **mixed parts** — pure whole, mixed parts, impossible for any table; measured by the **entanglement entropy** $S(\rho_A) = -\sum_i \lambda_i\log_2\lambda_i$ (eigenvalues of the part; $= S(\rho_B)$; $1$ bit for a Bell state; invariant under local gates $U\otimes V$) — **valid only when the whole is pure**: the coins have $S(\rho_A) = 1$ bit and no entanglement; a mixed state is **separable** if it is a mixture of products, and for two qubits the **partial transpose** decides it (a negative eigenvalue = entangled: Bell $-\tfrac12$, coins none); Bell and coins share every $Z$-statistic and differ in $X$: $\langle X\otimes X\rangle = 1$ vs $0$ |
| the general evolution of a probability vector is a **stochastic matrix** (Markov kernel) $p \to Tp$: columns are probability vectors, so probabilities stay non-negative and sum to $1$; a permutation is the reversible case; noise = a coin you did not see being tossed | a **quantum channel**: the general evolution $\rho \to \sum_k K_k\rho K_k^\dagger$ (Kraus operators, $\sum_k K_k^\dagger K_k = I$) — *derived* as "couple to an environment, evolve unitarily, trace the environment out" ($K_k = \langle k|_E\,U\,|0\rangle_E$); linear, trace-preserving and **completely** positive (the transpose is positive but sends half a Bell pair to a matrix with a $-\tfrac12$ eigenvalue — not a channel); a unitary is the reversible case ($K = U$), a stochastic matrix is the diagonal case ($K_{ij} = \sqrt{T_{ij}}\,|i\rangle\langle j|$); **decoherence = entanglement with an environment, then a partial trace**: dephasing (the environment learns the bit; coherence $\times e^{-t/T_\varphi}$), amplitude damping (energy leaks out; population $\times e^{-t/T_1}$, coherence $\times e^{-t/2T_1}$), depolarizing ($\rho \to (1-p)\rho + p\,I/2$); together they give $1/T_2 = 1/2T_1 + 1/T_\varphi$, hence $T_2 \le 2T_1$ — the laws `01/15` fitted |
| **Shannon entropy** $H(p) = -\sum_x p_x\log_2 p_x$: the average *surprise* $-\log_2 p_x$ (the log is forced by additivity of independent surprises); $0$ for a certain outcome, $\log_2 n$ for the uniform (Jensen), concave; **binary entropy** $h(p)$; joint and conditional entropy with the chain rule $H(AB) = H(A) + H(B|A)$, $H(B|A) \ge 0$, hence $H(A) \le H(AB)$ — a part is never more uncertain than the whole (coins: $1 = 1$; independent: $H(AB) = H(A) + H(B)$); the floor of lossless compression (Shannon 1948) | **von Neumann entropy** $S(\rho) = -\mathrm{tr}\,\rho\log_2\rho = H(\text{eigenvalues of }\rho)$ — the Shannon entropy of the eigenvalue coin (`01/13`): $0$ for a pure state, $1$ bit for $I/2$, $h\big(\tfrac{1 + |\vec r|}{2}\big)$ for a qubit; *but* the chain rule's positivity fails: the Bell state has $S(AB) = 0 < S(A) = 1$ (`01/18`) — a negative conditional entropy, `02/11`; the quantum side is earned in `02/09` |
| transport plan | quantum coupling |

(The last row is still forward-looking — earned in `04/04`.)

## What's next

Entropy measures *one* distribution. In `02/02_kl_divergence` we compare *two*: the relative entropy $D(p\|q)$ — the extra bits per symbol you pay for coding with the wrong model $q$ when the truth is $p$ (today's mismatched-code example, made exact), non-negative with equality only at $p = q$, and the single most important quantity in the rest of the course.

## References

- C. E. Shannon, *A mathematical theory of communication*, Bell Syst. Tech. J. **27**, 379–423 and 623–656 (1948) — §6 (the axioms that force $-\sum p\log p$; joint and conditional entropy, the chain rule, $H(A) \le H(AB)$), §9 (the source-coding theorem). doi:10.1002/j.1538-7305.1948.tb01338.x
- T. M. Cover & J. A. Thomas, *Elements of Information Theory*, 2nd ed., ch. 2 (entropy, joint and conditional entropy, chain rule, Jensen, concavity) and ch. 5 (Kraft inequality — Kraft 1949, McMillan 1956 — and source coding), Wiley (2006). doi:10.1002/047174882X
- A. I. Khinchin, *Mathematical Foundations of Information Theory*, Dover (1957) — the uniqueness theorem for the entropy function.
- D. J. C. MacKay, *Information Theory, Inference, and Learning Algorithms*, ch. 2 and 4–5, Cambridge University Press (2003) — freely available online; source coding by hand.

**Previous:** `notebooks/01_QuantumFoundations/19_quantum_channels.ipynb` · **Next:** `notebooks/02_InformationAndGeometry/02_kl_divergence.ipynb`